# Fundamentos de Arquitectura de Datos para IA/ML

**Objetivo**: Comprender los conceptos básicos de cómo diseñar arquitecturas de datos que soporten iniciativas de inteligencia artificial y aprendizaje automático.

---

## 1. Introducción: ¿Por qué la IA/ML requiere arquitecturas especiales?

Las aplicaciones tradicionales de datos se enfocan en almacenar, consultar y reportar información. Las aplicaciones de **IA y Machine Learning** tienen necesidades adicionales:

| Aspecto | Aplicación Tradicional | Aplicación de IA/ML |
|---------|------------------------|---------------------|
| **Volumen de datos** | Transacciones y reportes | Grandes datasets para entrenamiento |
| **Variedad** | Datos estructurados (tablas) | Estructurados + No estructurados (imágenes, texto, audio) |
| **Velocidad** | Batch diario/semanal | Tiempo real para inferencia |
| **Versionado** | Versión de esquema | Versión de datos + modelo + código |
| **Experimentación** | Consultas ad-hoc | Múltiples experimentos paralelos |
| **Gobernanza** | Datos y metadatos | Datos + modelos + linaje + reproducibilidad |

### Ejemplo real

Un sistema de **recomendación de productos** necesita:
- Datos históricos de compras (estructurados)
- Imágenes de productos (no estructurados)
- Comportamiento del usuario en tiempo real (streaming)
- Features preprocesadas listas para inferencia
- Múltiples versiones del modelo en producción

## 2. Ciclo de vida de ML y sus fases de datos

El desarrollo de soluciones de ML sigue un ciclo que requiere diferentes tipos de datos en cada fase:

```
┌─────────────┐     ┌──────────────┐     ┌─────────────┐     ┌──────────────┐
│  Ingesta    │────▶│ Preparación  │────▶│ Entrena-    │────▶│  Inferencia  │
│  de Datos   │     │  y Features  │     │  miento     │     │  (Producción)│
└─────────────┘     └──────────────┘     └─────────────┘     └──────────────┘
      │                    │                     │                    │
      │                    │                     │                    │
  Raw Data          Feature Store          Model Registry      Predictions
  Data Lake         (Online/Offline)       Versiones          Real-time API
```

### Fases clave

1. **Ingesta de Datos**
   - Recolección desde múltiples fuentes
   - Almacenamiento en Data Lake (raw)
   - Datos estructurados y no estructurados

2. **Preparación y Feature Engineering**
   - Limpieza y transformación
   - Creación de features (características)
   - Almacenamiento en Feature Store

3. **Entrenamiento**
   - Datasets de entrenamiento, validación y prueba
   - Experimentación con hiperparámetros
   - Versionado de modelos

4. **Inferencia/Producción**
   - Datos en tiempo real
   - Features online (baja latencia)
   - Predicciones y feedback loop

## 3. Componentes esenciales: Feature Store

Un **Feature Store** es un repositorio centralizado de features (características) procesadas y listas para usar en entrenamiento e inferencia.

### ¿Por qué es importante?

**Sin Feature Store:**
- Cada equipo recrea las mismas features
- Inconsistencia entre entrenamiento e inferencia
- Desperdicio de recursos computacionales
- Difícil mantener linaje de datos

**Con Feature Store:**
- Reutilización de features entre proyectos
- Consistencia garantizada
- Dos capas: Offline (entrenamiento) y Online (inferencia)
- Versionado y linaje automático

### Ejemplo conceptual

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from datetime import datetime

@dataclass
class Feature:
    """Representa una feature individual"""
    name: str
    value: Any
    timestamp: datetime
    version: str

@dataclass
class FeatureGroup:
    """Grupo de features relacionadas (ej: features de usuario)"""
    name: str
    features: List[Feature]
    entity_id: str  # ID del usuario, producto, etc.
    
class SimpleFeatureStore:
    """Simulación de un Feature Store básico"""
    
    def __init__(self):
        self.offline_storage = {}  # Para entrenamiento (batch)
        self.online_storage = {}   # Para inferencia (tiempo real)
    
    def write_offline(self, feature_group: FeatureGroup):
        """Escribe features para entrenamiento (batch)"""
        key = f"{feature_group.name}:{feature_group.entity_id}"
        self.offline_storage[key] = feature_group
        print(f"✓ Features guardadas en almacenamiento OFFLINE: {key}")
    
    def write_online(self, feature_group: FeatureGroup):
        """Escribe features para inferencia (baja latencia)"""
        key = f"{feature_group.name}:{feature_group.entity_id}"
        self.online_storage[key] = feature_group
        print(f"✓ Features guardadas en almacenamiento ONLINE: {key}")
    
    def get_training_data(self, feature_group_name: str) -> List[FeatureGroup]:
        """Obtiene features para entrenamiento"""
        results = [fg for key, fg in self.offline_storage.items() 
                   if fg.name == feature_group_name]
        print(f"→ Recuperadas {len(results)} feature groups de OFFLINE para entrenamiento")
        return results
    
    def get_online_features(self, feature_group_name: str, entity_id: str) -> FeatureGroup:
        """Obtiene features para inferencia (rápido)"""
        key = f"{feature_group_name}:{entity_id}"
        result = self.online_storage.get(key)
        print(f"→ Feature group recuperado de ONLINE para inferencia: {key}")
        return result

# Ejemplo de uso
print("=== Simulación de Feature Store ===")
print()

feature_store = SimpleFeatureStore()

# Crear features de usuario
user_features = FeatureGroup(
    name="user_profile_features",
    entity_id="user_12345",
    features=[
        Feature("age_group", "25-34", datetime.now(), "v1.0"),
        Feature("avg_purchase_value", 85.50, datetime.now(), "v1.0"),
        Feature("purchase_frequency_30d", 12, datetime.now(), "v1.0"),
        Feature("favorite_category", "electronics", datetime.now(), "v1.0")
    ]
)

# Escribir features para entrenamiento (OFFLINE)
feature_store.write_offline(user_features)

# Escribir features para inferencia (ONLINE)
feature_store.write_online(user_features)

print()
print("--- Escenario: Entrenamiento de modelo ---")
training_data = feature_store.get_training_data("user_profile_features")

print()
print("--- Escenario: Inferencia en producción ---")
online_features = feature_store.get_online_features("user_profile_features", "user_12345")

if online_features:
    print("\nFeatures disponibles para predicción:")
    for feat in online_features.features:
        print(f"  • {feat.name}: {feat.value}")

## 4. Tipos de datos en IA/ML

### 4.1 Datos Estructurados

Datos organizados en tablas con esquema definido:
- Transacciones de ventas
- Datos de sensores IoT
- Registros de usuarios

**Tecnologías**: SQL databases, Parquet, Delta Lake

### 4.2 Datos No Estructurados

Datos sin esquema predefinido:

| Tipo | Ejemplos | Casos de uso ML |
|------|----------|------------------|
| **Texto** | Emails, documentos, tweets | NLP, sentiment analysis, chatbots |
| **Imágenes** | Fotos, radiografías, satélites | Computer vision, clasificación |
| **Audio** | Llamadas, música, podcasts | Speech-to-text, reconocimiento |
| **Video** | Grabaciones, streaming | Detección de objetos, segmentación |

**Tecnologías**: Object Storage (S3, Azure Blob), Data Lakes

### 4.3 Datos Multimodales

Combinación de múltiples tipos de datos para un mismo análisis:

**Ejemplo: E-commerce**
- Descripción del producto (texto)
- Imagen del producto
- Reseñas de usuarios (texto)
- Historial de compras (estructurado)
- Videos de demostración

→ Modelo multimodal para recomendaciones más precisas

## 5. Diferencia entre datos de entrenamiento e inferencia

### Datos de Entrenamiento

**Características:**
- Históricos y batch
- Gran volumen (GB-TB)
- Incluyen labels (etiquetas) para aprendizaje supervisado
- Se procesan offline
- Latencia no crítica (puede tomar horas/días)

**Ejemplo**: Para entrenar un modelo de detección de fraude:
```python
training_data = {
    'transaction_id': [1, 2, 3, ...],  # Millones de registros
    'amount': [100, 2500, 50, ...],
    'merchant_category': ['retail', 'electronics', ...],
    'is_fraud': [0, 1, 0, ...]  # ← Label conocido
}
# Almacenado en Data Lake
# Procesado con Spark/Dask
```

### Datos de Inferencia

**Características:**
- En tiempo real o near-real-time
- Volumen pequeño por request
- Sin labels (es lo que queremos predecir)
- Requiere baja latencia (<100ms típicamente)
- Features deben estar precalculadas (Feature Store online)

**Ejemplo**: Detectar fraude en transacción actual:
```python
inference_request = {
    'transaction_id': 9999999,
    'amount': 5000,
    'merchant_category': 'jewelry',
    'is_fraud': ???  # ← Esto es lo que predecimos
}
# Respuesta requerida en milisegundos
# Features obtenidas de Feature Store online (Redis, DynamoDB)
```

## 6. Arquitectura de referencia simplificada

Una arquitectura básica para ML incluye estos componentes:

```
┌───────────────────────────────────────────────────────────────────┐
│                     CAPA DE INGESTA                                │
│  Batch: CSV, Parquet   │   Streaming: Kafka, Event Hub            │
└─────────────────┬─────────────────────────────────────────────────┘
                  │
                  ▼
┌───────────────────────────────────────────────────────────────────┐
│                     DATA LAKE (Raw)                                │
│  Estructurados: Parquet    │   No estructurados: S3/Blob          │
└─────────────────┬─────────────────────────────────────────────────┘
                  │
                  ▼
┌───────────────────────────────────────────────────────────────────┐
│               FEATURE ENGINEERING PIPELINE                         │
│  Spark / Pandas → Transformaciones → Agregaciones                 │
└─────────────────┬─────────────────────────────────────────────────┘
                  │
          ┌───────┴───────┐
          ▼               ▼
┌──────────────────┐  ┌──────────────────┐
│  FEATURE STORE   │  │  FEATURE STORE   │
│    OFFLINE       │  │     ONLINE       │
│  (Hive/Parquet)  │  │  (Redis/DynamoDB)│
└────────┬─────────┘  └────────┬─────────┘
         │                     │
         ▼                     ▼
┌──────────────────┐  ┌──────────────────┐
│  ENTRENAMIENTO   │  │   INFERENCIA     │
│  (Batch)         │  │   (Real-time)    │
│  → Model         │  │   → Predictions  │
└────────┬─────────┘  └──────────────────┘
         │
         ▼
┌──────────────────┐
│  MODEL REGISTRY  │
│  Versiones       │
│  Metadatos       │
└──────────────────┘
```

## 7. MLOps básico: concepto de versionado

En ML no basta con versionar el código, necesitas versionar tres elementos:

### 7.1 Versionado de Datos

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Dict

@dataclass
class DatasetVersion:
    """Representa una versión de dataset"""
    version: str
    path: str
    created_at: datetime
    num_samples: int
    schema_hash: str
    description: str

class DataVersionControl:
    """Simulación simple de versionado de datos"""
    
    def __init__(self):
        self.versions: Dict[str, DatasetVersion] = {}
    
    def register_dataset(self, version: str, path: str, num_samples: int, 
                         schema_hash: str, description: str):
        """Registra una nueva versión del dataset"""
        dataset_version = DatasetVersion(
            version=version,
            path=path,
            created_at=datetime.now(),
            num_samples=num_samples,
            schema_hash=schema_hash,
            description=description
        )
        self.versions[version] = dataset_version
        print(f"✓ Dataset v{version} registrado")
    
    def get_dataset(self, version: str) -> DatasetVersion:
        """Recupera una versión específica del dataset"""
        return self.versions.get(version)
    
    def list_versions(self):
        """Lista todas las versiones disponibles"""
        print("\n=== Versiones de Dataset Disponibles ===")
        for version, ds in sorted(self.versions.items()):
            print(f"\nv{version}:")
            print(f"  Muestras: {ds.num_samples:,}")
            print(f"  Fecha: {ds.created_at.strftime('%Y-%m-%d %H:%M')}")
            print(f"  Descripción: {ds.description}")
            print(f"  Path: {ds.path}")

# Ejemplo de uso
dvc = DataVersionControl()

# Registrar diferentes versiones
dvc.register_dataset(
    version="1.0",
    path="s3://ml-data/fraud_detection/v1.0/",
    num_samples=100000,
    schema_hash="abc123",
    description="Dataset inicial con datos de 2023"
)

dvc.register_dataset(
    version="1.1",
    path="s3://ml-data/fraud_detection/v1.1/",
    num_samples=150000,
    schema_hash="abc123",
    description="Agregados 50k samples de Q1-2024"
)

dvc.register_dataset(
    version="2.0",
    path="s3://ml-data/fraud_detection/v2.0/",
    num_samples=150000,
    schema_hash="def456",
    description="Cambio de esquema: agregadas features de geolocalización"
)

dvc.list_versions()

print("\n--- Recuperando dataset v1.1 para reproducir experimento ---")
dataset_v1_1 = dvc.get_dataset("1.1")
print(f"Recuperado: {dataset_v1_1.path}")

### 7.2 Versionado de Código y Modelos

**Código**:
- Git para scripts de entrenamiento
- Notebooks versionados
- Configuraciones de hiperparámetros

**Modelos**:
- Model Registry (MLflow, Azure ML, SageMaker)
- Artefactos binarios del modelo
- Metadatos: métricas, parámetros, dependencias

### Principio de reproducibilidad

Para reproducir un experimento necesitas:
```python
experiment = {
    'data_version': 'v1.1',          # ← Qué datos
    'code_commit': 'a3f5b2c',        # ← Qué código
    'model_version': 'v2.3',         # ← Qué modelo
    'hyperparameters': {             # ← Qué configuración
        'learning_rate': 0.001,
        'batch_size': 32
    }
}
# Con esta información puedes recrear exactamente el mismo resultado
```

## 8. Ejemplo simple: pipeline de datos para ML

Veamos un pipeline completo simplificado para un modelo de clasificación:

In [ ]:
import pandas as pd
from dataclasses import dataclass
from typing import List
import random

@dataclass
class MLPipeline:
    """Pipeline simplificado de datos para ML"""
    name: str
    
    def ingest_data(self, source: str) -> pd.DataFrame:
        """Paso 1: Ingesta de datos raw"""
        print(f"\n1️⃣  INGESTA desde {source}")
        # Simulación de datos de transacciones
        data = {
            'transaction_id': range(1, 101),
            'amount': [random.uniform(10, 1000) for _ in range(100)],
            'merchant_type': random.choices(['retail', 'online', 'restaurant'], k=100),
            'hour_of_day': [random.randint(0, 23) for _ in range(100)]
        }
        df = pd.DataFrame(data)
        print(f"   ✓ {len(df)} registros ingestados")
        return df
    
    def feature_engineering(self, df: pd.DataFrame) -> pd.DataFrame:
        """Paso 2: Creación de features"""
        print(f"\n2️⃣  FEATURE ENGINEERING")
        
        # Crear nuevas features
        df['amount_category'] = pd.cut(df['amount'], 
                                        bins=[0, 100, 500, 1000],
                                        labels=['low', 'medium', 'high'])
        
        df['is_night_transaction'] = (df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 6)
        
        # One-hot encoding de merchant_type
        df = pd.get_dummies(df, columns=['merchant_type'], prefix='merchant')
        
        print(f"   ✓ Features creadas: {list(df.columns)}")
        print(f"   ✓ Shape final: {df.shape}")
        return df
    
    def split_data(self, df: pd.DataFrame, train_ratio: float = 0.7):
        """Paso 3: División en train/validation/test"""
        print(f"\n3️⃣  SPLIT DE DATOS")
        
        n = len(df)
        train_size = int(n * train_ratio)
        val_size = int(n * 0.15)
        
        train_df = df.iloc[:train_size]
        val_df = df.iloc[train_size:train_size+val_size]
        test_df = df.iloc[train_size+val_size:]
        
        print(f"   ✓ Training set: {len(train_df)} samples ({len(train_df)/n*100:.1f}%)")
        print(f"   ✓ Validation set: {len(val_df)} samples ({len(val_df)/n*100:.1f}%)")
        print(f"   ✓ Test set: {len(test_df)} samples ({len(test_df)/n*100:.1f}%)")
        
        return train_df, val_df, test_df
    
    def save_to_feature_store(self, df: pd.DataFrame, storage_type: str):
        """Paso 4: Guardar en Feature Store"""
        print(f"\n4️⃣  GUARDADO EN FEATURE STORE ({storage_type})")
        if storage_type == "offline":
            print(f"   ✓ Guardado en Hive/Parquet para entrenamiento")
            print(f"   ✓ Path: s3://feature-store/fraud_detection/offline/")
        else:
            print(f"   ✓ Guardado en Redis para inferencia en tiempo real")
            print(f"   ✓ TTL: 24 horas, actualización cada hora")
    
    def run(self, source: str):
        """Ejecuta el pipeline completo"""
        print(f"\n{'='*60}")
        print(f"Pipeline: {self.name}")
        print(f"{'='*60}")
        
        # Paso 1: Ingesta
        df = self.ingest_data(source)
        
        # Paso 2: Feature engineering
        df_features = self.feature_engineering(df)
        
        # Paso 3: Split
        train_df, val_df, test_df = self.split_data(df_features)
        
        # Paso 4: Guardar en Feature Store
        self.save_to_feature_store(train_df, "offline")
        self.save_to_feature_store(df_features.tail(10), "online")
        
        print(f"\n✅ Pipeline completado exitosamente")
        print(f"{'='*60}")
        
        return train_df, val_df, test_df

# Ejecutar el pipeline
pipeline = MLPipeline(name="Fraud Detection Data Pipeline")
train_df, val_df, test_df = pipeline.run(source="transactions_db")

## 9. Requisitos de infraestructura

Para soportar arquitecturas AI-ready necesitas:

### Almacenamiento

| Componente | Tecnologías | Propósito |
|------------|-------------|------------|
| **Data Lake** | S3, Azure Blob, GCS | Datos raw (estructurados + no estructurados) |
| **Feature Store Offline** | Hive, Delta Lake, Parquet | Features para entrenamiento (batch) |
| **Feature Store Online** | Redis, DynamoDB, Cassandra | Features para inferencia (baja latencia) |
| **Model Registry** | MLflow, Kubeflow, SageMaker | Modelos versionados |

### Cómputo

| Fase | Recursos | Justificación |
|------|----------|---------------|
| **Feature Engineering** | Spark, Dask clusters | Procesamiento distribuido de gran volumen |
| **Entrenamiento** | GPUs (V100, A100) | Aceleración de deep learning |
| **Inferencia Batch** | CPUs, clusters Kubernetes | Predicciones por lotes |
| **Inferencia Real-time** | APIs (FastAPI), serverless | Baja latencia, alta disponibilidad |

### Orquestación

- **Airflow**: Orquestación de pipelines de datos
- **Kubeflow**: Orquestación de workflows de ML
- **MLflow**: Tracking de experimentos y deployment

## 10. Checklist: ¿Tu arquitectura está AI-ready?

Evalúa tu arquitectura actual con esta lista:

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class AIReadinessCheck:
    """Evaluador de preparación para IA/ML"""
    category: str
    questions: List[str]
    
class AIReadinessAssessment:
    """Evaluación de madurez AI-ready"""
    
    def __init__(self):
        self.checks = [
            AIReadinessCheck(
                category="📊 Datos",
                questions=[
                    "¿Tienen un Data Lake para almacenar datos raw?",
                    "¿Pueden ingestar datos en streaming (tiempo real)?",
                    "¿Almacenan datos no estructurados (imágenes, texto)?",
                    "¿Tienen control de versiones de datasets?"
                ]
            ),
            AIReadinessCheck(
                category="⚙️ Feature Engineering",
                questions=[
                    "¿Tienen un Feature Store implementado?",
                    "¿Las features están disponibles para entrenamiento E inferencia?",
                    "¿Existe linaje de features (de dónde vienen)?",
                    "¿Las features son reutilizables entre proyectos?"
                ]
            ),
            AIReadinessCheck(
                category="🤖 MLOps",
                questions=[
                    "¿Tienen un Model Registry para versionar modelos?",
                    "¿Existe un pipeline automatizado de entrenamiento?",
                    "¿Pueden reproducir experimentos anteriores?",
                    "¿Monitorean modelos en producción (drift, performance)?"
                ]
            ),
            AIReadinessCheck(
                category="🚀 Inferencia",
                questions=[
                    "¿Pueden servir modelos con latencia <100ms?",
                    "¿Tienen APIs de inferencia en producción?",
                    "¿Existe un sistema de feedback loop (predicciones → datos)?",
                    "¿Pueden escalar inferencia según demanda?"
                ]
            ),
            AIReadinessCheck(
                category="🔒 Gobernanza",
                questions=[
                    "¿Tienen linaje completo de datos para modelos?",
                    "¿Existe control de acceso a datos sensibles?",
                    "¿Los modelos son auditables (explainability)?",
                    "¿Cumplen regulaciones (GDPR, bias detection)?"
                ]
            )
        ]
    
    def run_assessment(self, responses: List[List[bool]]) -> dict:
        """Ejecuta la evaluación y retorna el score"""
        results = {}
        total_yes = 0
        total_questions = 0
        
        print("\n" + "="*70)
        print("📋 EVALUACIÓN DE MADUREZ AI-READY")
        print("="*70)
        
        for i, check in enumerate(self.checks):
            category_responses = responses[i]
            yes_count = sum(category_responses)
            total = len(category_responses)
            percentage = (yes_count / total) * 100
            
            results[check.category] = {
                'score': percentage,
                'yes_count': yes_count,
                'total': total
            }
            
            total_yes += yes_count
            total_questions += total
            
            print(f"\n{check.category}")
            print(f"  Score: {percentage:.0f}% ({yes_count}/{total})")
            
            # Mostrar estado
            if percentage >= 75:
                status = "✅ Excelente"
            elif percentage >= 50:
                status = "⚠️  Necesita mejora"
            else:
                status = "❌ Crítico"
            print(f"  Estado: {status}")
        
        overall_score = (total_yes / total_questions) * 100
        
        print("\n" + "="*70)
        print(f"🎯 SCORE GENERAL: {overall_score:.0f}% ({total_yes}/{total_questions})")
        print("="*70)
        
        # Nivel de madurez
        if overall_score >= 80:
            level = "🌟 Nivel 4: AI-Ready Avanzado"
            recommendation = "Su arquitectura está lista para iniciativas de IA a escala."
        elif overall_score >= 60:
            level = "🔧 Nivel 3: En desarrollo"
            recommendation = "Tiene fundamentos sólidos. Priorice Feature Store y MLOps."
        elif overall_score >= 40:
            level = "⚡ Nivel 2: Inicial"
            recommendation = "Comience con un Data Lake y pipelines de feature engineering."
        else:
            level = "🚧 Nivel 1: Pre-AI"
            recommendation = "Enfóquese primero en arquitectura de datos básica."
        
        print(f"\n{level}")
        print(f"💡 Recomendación: {recommendation}\n")
        
        return results

# Ejemplo de uso
assessment = AIReadinessAssessment()

# Simular respuestas de una organización
# [Datos, Feature Engineering, MLOps, Inferencia, Gobernanza]
responses = [
    [True, True, False, False],   # Datos: 2/4
    [False, False, False, True],  # Feature Engineering: 1/4
    [True, False, False, False],  # MLOps: 1/4
    [True, True, False, True],    # Inferencia: 3/4
    [False, True, False, False]   # Gobernanza: 1/4
]

results = assessment.run_assessment(responses)

## Conclusión

### Puntos clave aprendidos

1. **IA/ML requiere arquitecturas especializadas** que soporten:
   - Datos estructurados + no estructurados + multimodales
   - Procesamiento batch (entrenamiento) + real-time (inferencia)
   - Versionado de datos, código y modelos

2. **Feature Store es el componente central**:
   - Reutilización de features
   - Consistencia entre entrenamiento e inferencia
   - Dos capas: offline (batch) y online (tiempo real)

3. **El ciclo de vida de ML tiene 4 fases**:
   - Ingesta → Feature Engineering → Entrenamiento → Inferencia
   - Cada fase requiere diferentes tecnologías y patrones

4. **MLOps es fundamental**:
   - Versionado triple (datos + código + modelos)
   - Reproducibilidad de experimentos
   - Automatización de pipelines

### Próximos pasos

En el siguiente nivel (Mid) profundizaremos en:
- Diseño detallado de arquitecturas event-driven para ML
- Implementación de Feature Store (offline/online)
- Caso práctico: sistema de recomendación en streaming completo
- Integración de datos multimodales (texto + imágenes)

---

**Referencias**:
- "Designing Machine Learning Systems" - Chip Huyen
- "Building Machine Learning Powered Applications" - Emmanuel Ameisen
- MLOps.org Community Guidelines